# Scaffolding project

_DSAIT4050: Information retrieval lecture, TU Delft_

Welcome to the **DSAIT4050: Information retrieval** lecture!

This project acts as a gentle introduction to information retrieval for you. You do not need any prior knowledge about IR for this task. Only some Python programming skills are required.

## Getting started

Under the hood, this notebook uses a library called **PyTerrier**. Please check out the first part of our _Introduction to PyTerrier_ series to learn how to install PyTerrier. However, you do not need to interact with PyTerrier directly for now; rather, we're providing you with simple utility functions you can use. Feel free to have a look how these are implemented, but it's not required.

**Task 1**: Install PyTerrier (see the `01-setup.ipynb` notebook).

Now you should be able to import the utility functions. A dataset will be downloaded and indexed automatically (this will take a minute).


In [2]:
pip install python-terrier==0.12.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
from util import search, evaluate, evaluate_all

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
antique/test/non-offensive documents: 100%|██████████| 403666/403666 [01:04<00:00, 6294.75it/s]


Now that we have loaded the data, you can run search queries. For example:


In [4]:
search("what is the meaning of life")

17:58:06.188 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of ''. Set a termpipelines control to remove this warning.


,qid,docid,docno,text,rank,score,query
0,1,284036,327334_2,If life did not suck sometime it would not be ...,0,25.241914,what is the meaning of life
1,1,24563,514843_5,"To live until we die,and have a meaningful. li...",1,24.428920,what is the meaning of life
2,1,183199,3286609_30,To make my mom and dad's life meaningful.,2,24.428920,what is the meaning of life
3,1,338875,1977360_8,"Change it to "" What makes life meaningful?""",3,24.428920,what is the meaning of life
4,1,146534,422602_7,to have a meaningful life or a life of meaning...,4,22.001176,what is the meaning of life
5,1,37272,3770850_1,"Oh... I get it.... ""thes""?. . Thes. is the abb...",5,21.882860,what is the meaning of life
6,1,142099,3352535_8,whats the purpose of life?,6,21.260102,what is the meaning of life
7,1,169083,2534143_13,whats life with no exitment!,7,21.260102,what is the meaning of life
8,1,338872,1977360_5,"Perhaps ""what can you do to make your life mor...",8,21.040540,what is the meaning of life
9,1,104996,4351526_3,the meaning of life is whatever you ascribe to...,9,19.934543,what is the meaning of life


What you get here is a list of ten documents from the corpus that are ordered by how relevant they are to our query (according to the search engine).

## Query rewriting

The goal of this task is to come up with a way of **rewriting queries** such that the search engine can "understand" them better.

In order to do this, let's first take a look at some example queries from our dataset. We represent these queries using a `pandas.DataFrame`, where the first column corresponds to the **query ID** and the second column corresponds to the **query**:


In [5]:
import pandas as pd

example_queries = pd.DataFrame(
    [
        [
            "443848",
            "does anybody know where i could get a free guide on how to train a siberian husky",
        ],
        [
            "1783010",
            "what is blaphsemy",
        ],
        [
            "2838988",
            "how can i get a cork out of not into a wine bottle without a corkscrew",
        ],
    ],
    columns=["qid", "query"],
)

Since these queries are taken from the dataset, we can **evaluate the performance** of our search engine on these queries. This means that we know which documents the system should retrieve for each query.

You can use the following evaluation function to do this. This function takes your queries and returns a score (mean average precision -- you will learn about this later). For now, all you need to know is that, the higher this score, the better the system works.

Let's evaluate the queries we have:


In [6]:
print("score:", evaluate(example_queries))

score: 0.07906002902973568


Now it's up to you to figure out if and how it's possible to make the search engine perform better on these queries. How would you query a search engine if you wanted to know about these topics? Experiment a bit.

**Task 2**: Try to manually come up with ways to rewrite or reformulate the queries so the performance improves.

**Important**: Make sure that the query IDs match! Otherwise, evaluation will not work.


In [8]:
example_queries_rewritten = pd.DataFrame(
    [
        # Write reflection about method used
        [
            "443848",
            # Original: does anybody know where i could get a free guide on how to train a siberian husky
            # Adding more general terms such as "Dog" helps find more relevant results
            "siberian husky dog train",
            #"siberian husky train dog",
        ],
        [
            "1783010",
            # Original: blaphsemy
            # Fix typos
            "what is Blasphemy",
            #"what Blasphemy is",
        ],
        [
            "2838988",
            # Original: how can i get a cork out of not into a wine bottle without a corkscrew
            "cork bottle without corkscrew",
            #"bottle corkscrew cork without",
        ],
    ],
    columns=["qid", "query"],
)
print("score after rewriting:", evaluate(example_queries_rewritten))

score after rewriting: 0.10694392789340007


In [15]:
!pip install pyspellchecker
!pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 20.1 MB/s eta 0:00:00


# An automatic approach

In this last part, we'll try to come up with an automatic approach to perform query re-writing. Use your findings from task 2 for this.

**Task 3**: Implement a function that automatically re-writes any input query.

You can use any approach or library you want for this task. However, keep in mind that simple ideas often work well!


In [26]:
import re
from spellchecker import SpellChecker
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')
custom_stopwords = {'something'}
stop_words = set(stopwords.words('english')).union(custom_stopwords)
spell = SpellChecker()


def remove_typos(query: str) -> str:
    """
    Remove typos using spellchecker.
    """
    words = query.split(" ")
    misspelled_words = spell.unknown(words)
    corrected_words = []
    for word in words:
        if word in misspelled_words:
            corrected_word = spell.correction(word)
            if corrected_word is not None:
                corrected_words.append(corrected_word)
        else:
            corrected_words.append(word)
    return " ".join(corrected_words)


def format_query(query: str) -> str:
    """
    Formats a query.
    Removes capitalization, unnecessary white spaces and newlines.
    """
    # Remove punctuation
    no_punctuation = re.sub(r'[\.,!?;:\'"\-\(\)\[\]\{\}<>]', '', query)

    # Remove newlines and replace them with a single space
    no_newlines = re.sub(r'\n', ' ', no_punctuation)

    # Normalize whitespace to a single space
    normalized_whitespace = re.sub(r'\s+', ' ', no_newlines)

    # Convert to lowercase
    cleaned_string = normalized_whitespace.strip().lower()

    return cleaned_string


def should_keep_word(word: str) -> bool:
    """
    Whether a word should be kept or not.
    """
    is_stopword = word in stop_words
    return not is_stopword


def filter_words(query: str) -> str:
    """
    Remove unnecessary words from query.
    """
    return " ".join([split_word for split_word in query.split(" ") if split_word != " " and should_keep_word(split_word)])


def add_words(query: str) -> str:
    """
    Add words to query.
    """
    # TODO write function
    return query


def rewrite_query(query: str) -> str:
    formatted_query = format_query(query)
    formatted_query = remove_typos(formatted_query)
    formatted_query = add_words(formatted_query)
    formatted_query = filter_words(formatted_query)

    formatted_optimized_query = format_query(formatted_query)
    print(f"optimized query: {formatted_optimized_query}")
    return formatted_optimized_query

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\maxde\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


This time, we'll evalute on _all_ queries in the dataset. This will give us a more general result:


In [20]:
print("score:", evaluate_all())

score: 0.06179994498738492


Are you able to improve the overall performance using your rewriting approach?


In [27]:
print("score after rewriting", evaluate_all(rewrite_query))

optimized query: get concentration
optimized query: water fall earth round
optimized query: determine charge iron ion feel
optimized query: mice get rid humanely
optimized query: see leaflet mean eat pregnancy test
optimized query: innate immunity
optimized query: lose 30 pounds june
optimized query: words write sound raindrops moving train scribbling w pencil paper figuratively
optimized query: must cracked windshield order car pass safety inspection
optimized query: blasphemy
optimized query: proper way express sorrow funeral
optimized query: speculate happened natalie holloway aruba
optimized query: people get hiccups exactly best way cure
optimized query: elected federal senators congressmen live regular like rest us
optimized query: people judge dog looks like
optimized query: word remission mean referring cancer patients
optimized query: patient driver
optimized query: ego part survival instincts
optimized query: cats headbutt
optimized query: cook angus burgers
optimized query: 

C:\Users\maxde\anaconda3\envs\ms-information-retrieval\lib\site-packages\pyterrier\terrier\retriever.py:282: UserWarning: Skipping empty query for qid 224109
  warn(
C:\Users\maxde\anaconda3\envs\ms-information-retrieval\lib\site-packages\pyterrier\terrier\retriever.py:282: UserWarning: Skipping empty query for qid 78762
  warn(


score after rewriting 0.07760985398223355
